In [ ]:
reg.exe query "HKLM\SOFTWARE\Microsoft\MSBuild\ToolsVersions\4.0" /v MSBuildToolsPath

reg.exe query "HKLM\SOFTWARE\Microsoft\MSBuild\ToolsVersions\14.0" /v MSBuildToolsPath



In [ ]:
# Define the registry paths we need to lock down
$Paths = @(
    "Registry::HKEY_CLASSES_ROOT\.txt",
    "Registry::HKEY_CLASSES_ROOT\.txt\ShellNew"
)

foreach ($Path in $Paths) {
    # Get the existing security permissions of the key
    $Acl = Get-Acl -Path $Path
    
    # Create a rule that explicitly DENIES "Write" and "Create" permissions to Everyone
    $TargetUser = New-Object System.Security.Principal.NTAccount("Everyone")
    $Rights = [System.Security.AccessControl.RegistryRights]::SetValue -bor [System.Security.AccessControl.RegistryRights]::CreateSubKey
    $Inheritance = [System.Security.AccessControl.InheritanceFlags]::None
    $Propagation = [System.Security.AccessControl.PropagationFlags]::None
    $Type = [System.Security.AccessControl.AccessControlType]::Deny
    
    $Rule = New-Object System.Security.AccessControl.RegistryAccessRule($TargetUser, $Rights, $Inheritance, $Propagation, $Type)
    
    # Apply the absolute Deny rule to the key
    $Acl.AddAccessRule($Rule)
    Set-Acl -Path $Path -AclObject $Acl
}

Write-Host "Registry paths successfully locked down to Read-Only!" -ForegroundColor Green
